In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

# Project root
BASE_DIR = Path.cwd().parent
RAW_DIR = BASE_DIR / 'data' / 'raw_data'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed_data'
PROCESSED_DIR.mkdir(exist_ok=True)

print("Raw data folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)

Raw data folder: C:\Users\DELL\OneDrive\Desktop\bluestock_mf_capstone\data\raw_data
Processed folder: C:\Users\DELL\OneDrive\Desktop\bluestock_mf_capstone\data\processed_data


In [2]:
nav = pd.read_csv(RAW_DIR / '02_nav_history.csv')

# Fix date, sort, forward-fill weekends/holidays
nav['date'] = pd.to_datetime(nav['date'])
nav = nav.sort_values(['amfi_code', 'date'])
nav = nav[nav['nav'] > 0]  # remove invalid NAVs
nav = nav.drop_duplicates()

# Add daily return column
nav['daily_return_pct'] = nav.groupby('amfi_code')['nav'].pct_change() * 100

nav.to_csv(PROCESSED_DIR / 'clean_nav.csv', index=False)
print(f"✅ clean_nav.csv — {nav.shape[0]} rows")
print(nav.head(3))

✅ clean_nav.csv — 46000 rows
      amfi_code       date       nav  daily_return_pct
5750     100016 2022-01-03  520.4608               NaN
5751     100016 2022-01-04  515.0971         -1.030568
5752     100016 2022-01-05  521.7239          1.286515


In [3]:
txn = pd.read_csv(RAW_DIR / '08_investor_transactions.csv')

# Fix date
txn['transaction_date'] = pd.to_datetime(txn['transaction_date'])

# Standardise transaction type
txn['transaction_type'] = txn['transaction_type'].str.strip().str.title()
print("Transaction types:", txn['transaction_type'].unique())

# Remove invalid amounts
txn = txn[txn['amount_inr'] > 0]

# Remove duplicates
txn = txn.drop_duplicates()

txn.to_csv(PROCESSED_DIR / 'clean_transactions.csv', index=False)
print(f"✅ clean_transactions.csv — {txn.shape[0]} rows")

Transaction types: ['Sip' 'Redemption' 'Lumpsum']
✅ clean_transactions.csv — 32778 rows


In [4]:
perf = pd.read_csv(RAW_DIR / '07_scheme_performance.csv')

# Validate numeric columns
num_cols = ['return_1yr_pct','return_3yr_pct','return_5yr_pct',
            'sharpe_ratio','beta','alpha','expense_ratio_pct']
for col in num_cols:
    perf[col] = pd.to_numeric(perf[col], errors='coerce')

# Flag bad expense ratios
bad = perf[(perf['expense_ratio_pct'] < 0.1) | (perf['expense_ratio_pct'] > 2.5)]
if len(bad) > 0:
    print(f"⚠️ {len(bad)} rows with unusual expense ratio")
else:
    print("✅ All expense ratios in valid range (0.1% – 2.5%)")

# Flag negative Sharpe
neg_sharpe = perf[perf['sharpe_ratio'] < 0]
print(f"⚠️ Negative Sharpe ratios: {len(neg_sharpe)}")

perf.to_csv(PROCESSED_DIR / 'clean_performance.csv', index=False)
print(f"✅ clean_performance.csv — {perf.shape[0]} rows")

✅ All expense ratios in valid range (0.1% – 2.5%)
⚠️ Negative Sharpe ratios: 0
✅ clean_performance.csv — 40 rows


In [5]:
other_files = {
    '01_fund_master.csv':         'clean_fund_master.csv',
    '03_aum_by_fund_house.csv':   'clean_aum.csv',
    '04_monthly_sip_inflows.csv': 'clean_sip_inflows.csv',
    '05_category_inflows.csv':    'clean_category_inflows.csv',
    '06_industry_folio_count.csv':'clean_folio_count.csv',
    '09_portfolio_holdings.csv':  'clean_portfolio_holdings.csv',
    '10_benchmark_indices.csv':   'clean_benchmark_indices.csv',
}

date_cols = {
    '01_fund_master.csv':         'launch_date',
    '03_aum_by_fund_house.csv':   'date',
    '04_monthly_sip_inflows.csv': 'month',
    '05_category_inflows.csv':    'month',
    '06_industry_folio_count.csv':'month',
    '09_portfolio_holdings.csv':  'portfolio_date',
    '10_benchmark_indices.csv':   'date',
}

for raw_file, clean_file in other_files.items():
    df = pd.read_csv(RAW_DIR / raw_file)
    df = df.drop_duplicates()
    if raw_file in date_cols:
        df[date_cols[raw_file]] = pd.to_datetime(df[date_cols[raw_file]])
    df.to_csv(PROCESSED_DIR / clean_file, index=False)
    print(f"✅ {clean_file} — {df.shape[0]} rows")

✅ clean_fund_master.csv — 40 rows
✅ clean_aum.csv — 90 rows
✅ clean_sip_inflows.csv — 48 rows
✅ clean_category_inflows.csv — 144 rows
✅ clean_folio_count.csv — 21 rows
✅ clean_portfolio_holdings.csv — 322 rows
✅ clean_benchmark_indices.csv — 8050 rows


In [6]:
print("\n=== CLEANED FILES SUMMARY ===")
for f in sorted(PROCESSED_DIR.glob('*.csv')):
    df = pd.read_csv(f)
    print(f"{f.name}: {df.shape[0]} rows × {df.shape[1]} cols")


=== CLEANED FILES SUMMARY ===
clean_aum.csv: 90 rows × 5 cols
clean_benchmark_indices.csv: 8050 rows × 3 cols
clean_category_inflows.csv: 144 rows × 3 cols
clean_folio_count.csv: 21 rows × 6 cols
clean_fund_master.csv: 40 rows × 15 cols
clean_nav.csv: 46000 rows × 4 cols
clean_performance.csv: 40 rows × 19 cols
clean_portfolio_holdings.csv: 322 rows × 8 cols
clean_sip_inflows.csv: 48 rows × 6 cols
clean_transactions.csv: 32778 rows × 13 cols


In [7]:
from sqlalchemy import create_engine
import sqlite3

DB_PATH = BASE_DIR / 'data' / 'db' / 'bluestock_mf.db'
engine = create_engine(f'sqlite:///{DB_PATH}')

# Load each cleaned CSV into the database
table_map = {
    'clean_fund_master.csv':        'dim_fund',
    'clean_nav.csv':                'fact_nav',
    'clean_transactions.csv':       'fact_transactions',
    'clean_performance.csv':        'fact_performance',
    'clean_aum.csv':                'fact_aum',
    'clean_sip_inflows.csv':        'fact_sip_industry',
    'clean_portfolio_holdings.csv': 'fact_portfolio',
    'clean_benchmark_indices.csv':  'fact_benchmark',
}

for csv_file, table_name in table_map.items():
    df = pd.read_csv(PROCESSED_DIR / csv_file)
    df.to_sql(table_name, engine, if_exists='replace', index=False)
    print(f"✅ Loaded {len(df)} rows → {table_name}")

print("\n✅ Database created successfully!")

✅ Loaded 40 rows → dim_fund
✅ Loaded 46000 rows → fact_nav
✅ Loaded 32778 rows → fact_transactions
✅ Loaded 40 rows → fact_performance
✅ Loaded 90 rows → fact_aum
✅ Loaded 48 rows → fact_sip_industry
✅ Loaded 322 rows → fact_portfolio
✅ Loaded 8050 rows → fact_benchmark

✅ Database created successfully!


In [8]:
conn = sqlite3.connect(DB_PATH)

tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print("Tables in database:")
for t in tables['name']:
    count = pd.read_sql(f"SELECT COUNT(*) as rows FROM {t}", conn)
    print(f"  {t}: {count['rows'][0]} rows")

conn.close()

Tables in database:
  dim_fund: 40 rows
  fact_nav: 46000 rows
  fact_transactions: 32778 rows
  fact_performance: 40 rows
  fact_aum: 90 rows
  fact_sip_industry: 48 rows
  fact_portfolio: 322 rows
  fact_benchmark: 8050 rows


In [11]:
conn = sqlite3.connect(DB_PATH)

queries = {
    "Q1: Top 5 funds by AUM": """
        SELECT scheme_name, fund_house, aum_crore
        FROM fact_performance ORDER BY aum_crore DESC LIMIT 5""",

    "Q2: Avg NAV per month - SBI Bluechip": """
    SELECT strftime('%Y-%m', date) AS month, ROUND(AVG(nav),2) AS avg_nav
    FROM fact_nav WHERE amfi_code = 119551
    GROUP BY month ORDER BY month""",

    "Q3: SIP inflow YoY growth": """
        SELECT month, sip_inflow_crore, yoy_growth_pct
        FROM fact_sip_industry WHERE yoy_growth_pct IS NOT NULL
        ORDER BY month""",

    "Q4: Transaction amount by state": """
        SELECT state, ROUND(SUM(amount_inr)/10000000.0, 2) AS total_crore
        FROM fact_transactions GROUP BY state
        ORDER BY total_crore DESC""",

    "Q5: Funds with expense ratio < 1%": """
        SELECT scheme_name, fund_house, expense_ratio_pct
        FROM dim_fund WHERE expense_ratio_pct < 1.0
        ORDER BY expense_ratio_pct ASC""",

    "Q6: Top 5 funds by Sharpe ratio": """
        SELECT scheme_name, sharpe_ratio, risk_grade
        FROM fact_performance ORDER BY sharpe_ratio DESC LIMIT 5""",

    "Q7: Avg SIP amount by age group": """
    SELECT age_group, ROUND(AVG(amount_inr),2) AS avg_sip,
    COUNT(*) AS total_txns
    FROM fact_transactions WHERE UPPER(transaction_type) = 'SIP'
    GROUP BY age_group ORDER BY avg_sip DESC""",

    "Q8: Fund count per category": """
        SELECT category, sub_category, COUNT(*) AS num_funds
        FROM dim_fund GROUP BY category, sub_category
        ORDER BY num_funds DESC""",

    "Q9: Top 5 sectors by portfolio weight": """
        SELECT sector, ROUND(SUM(weight_pct),2) AS total_weight
        FROM fact_portfolio GROUP BY sector
        ORDER BY total_weight DESC LIMIT 5""",

    "Q10: Benchmark index performance": """
        SELECT index_name, ROUND(MAX(close_value),2) AS latest,
        ROUND((MAX(close_value)-MIN(close_value))/MIN(close_value)*100,2) AS growth_pct
        FROM fact_benchmark GROUP BY index_name
        ORDER BY growth_pct DESC""",
}

for title, query in queries.items():
    print(f"\n{'='*50}")
    print(f"📊 {title}")
    print('='*50)
    result = pd.read_sql(query, conn)
    print(result.to_string(index=False))

conn.close()
print("\n✅ All 10 queries executed successfully!")


📊 Q1: Top 5 funds by AUM
                                          scheme_name        fund_house  aum_crore
Mirae Asset Emerging Bluechip Fund - Regular - Growth    Mirae Asset MF      49046
        Kotak Emerging Equity Fund - Regular - Growth Kotak Mahindra MF      47469
       Nippon India Small Cap Fund - Regular - Growth   Nippon India MF      43630
           DSP Top 100 Equity Fund - Regular - Growth   DSP Mutual Fund      41828
                  UTI Mid Cap Fund - Regular - Growth   UTI Mutual Fund      41728

📊 Q2: Avg NAV per month - SBI Bluechip
  month  avg_nav
2022-01    55.09
2022-02    52.48
2022-03    50.84
2022-04    52.04
2022-05    51.89
2022-06    52.68
2022-07    52.69
2022-08    53.77
2022-09    56.55
2022-10    57.88
2022-11    60.85
2022-12    61.35
2023-01    60.27
2023-02    61.13
2023-03    64.59
2023-04    67.36
2023-05    66.21
2023-06    69.62
2023-07    70.83
2023-08    73.76
2023-09    74.03
2023-10    71.21
2023-11    72.89
2023-12    72.19
2024-01    